In [ ]:
import base64
import json
from openai import OpenAI

client = OpenAI(api_key="INSERT_KEY")



In [14]:
from PIL import Image
import numpy as np


def tiff_to_png(input_path: str, output_path: str):
    img = Image.open(input_path)

    # Convert to numpy for normalization if needed
    arr = np.array(img)

    # Handle grayscale normalization (important for microscopy)
    if arr.dtype != np.uint8:
        arr = arr.astype(np.float32)
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
        arr = (arr * 255).astype(np.uint8)

    # Convert back to image
    img_out = Image.fromarray(arr)

    # Ensure correct mode
    if img_out.mode not in ["L", "RGB"]:
        img_out = img_out.convert("L")

    img_out.save(output_path)

import glob
from pathlib import Path
all_files = glob.glob("data/separated_frames/*.tiff")
for i in all_files:
    path = i
    path = Path(path)
    tiff_to_png(path, f"data/pngs/{path.stem}.png")

In [ ]:

def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def detect_filament(image_path):
    image_b64 = encode_image(image_path)

    prompt = """
Determine whether this image contains at least one filament.

A filament is a thin, elongated structure.
Filaments will be visible as a thin, elongated high-density structure within a cell whereas cell textures
will present as circular forms with moderate density.
A high aspect ratio is characteristic of a filament.
Do NOT count diffuse cell texture or broad structures.

Return ONLY valid JSON:
{"contains_filament": true/false, "confidence": 0-1, "reason": "..."}
"""

    response = client.responses.create(
        model="gpt-4.1",  # or "gpt-4o"
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/png;base64,{image_b64}",
                    },
                ],
            }
        ],
        temperature=0,
    )

    text = response.output[0].content[0].text.strip()

    # Safe JSON parsing
    start = text.find("{")
    end = text.rfind("}")
    return json.loads(text[start:end + 1])


if __name__ == "__main__":
    all_files = glob.glob("data/pngs/*.png")
    for i in all_files[:10]:
        result = detect_filament(i)
        print(result)
        print("Filament present:", result["contains_filament"])

{'contains_filament': False, 'confidence': 0.85, 'reason': 'The image shows a broad, somewhat circular high-density structure with no clear thin, elongated, high aspect ratio features characteristic of filaments. The visible structure is more consistent with diffuse cell texture.'}
Filament present: False
{'contains_filament': False, 'confidence': 0.95, 'reason': 'The image shows a broad, dense, and somewhat circular structure without any visible thin, elongated high-density features characteristic of filaments. The observed structure does not have a high aspect ratio and appears more like diffuse cell texture.'}
Filament present: False
{'contains_filament': False, 'confidence': 0.85, 'reason': 'The visible structures are more circular and broad rather than thin and elongated. There is no clear high aspect ratio structure characteristic of a filament; the bright area appears as part of a rounded cell texture.'}
Filament present: False
{'contains_filament': True, 'confidence': 0.85, 're